# RAG Assignment: Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking

This experiment uses **NIST SP 800-63B-4, Digital Identity Guidelines: Authentication and Authenticator Management** (July 2025), an official, publicly available NIST technical standard. The PDF is downloaded from NIST's publication server; the assignment-instructions PDF is not used as corpus data.

The notebook extracts page text from the source PDF, retains PDF page numbers, and creates overlapping chunks programmatically. Both pipelines use the same chunk corpus: the bi-encoder retrieves the top 10; Pipeline 1 selects its top 3 and Pipeline 2 re-ranks those same 10 with a cross-encoder before selecting its top 3. Queries are grounded in clauses present in the PDF. Original ranks, re-ranked positions, selected evidence, scores, answers, and measured timings are reported as observed; no expected rank shifts are imposed.

In [1]:
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SOURCE_PDF_NAME = 'NIST_SP_800-63B-4_Digital_Identity_Guidelines.pdf'
SOURCE_URL = 'https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-63b-4.pdf'
EMBEDDER_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
RERANKER_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
GENERATOR_NAME = 'google/flan-t5-base'

pdf_path = Path.cwd() / SOURCE_PDF_NAME
if not pdf_path.is_file():
    raise FileNotFoundError(f'Source PDF not found in notebook working directory: {pdf_path}')
print(f'Source PDF: {pdf_path.name}')
print(f'Official source: {SOURCE_URL}')

Source PDF: NIST_SP_800-63B-4_Digital_Identity_Guidelines.pdf
Official source: https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-63b-4.pdf


In [2]:
reader = PdfReader(str(pdf_path))
if reader.is_encrypted:
    raise ValueError('The source PDF is encrypted and cannot be extracted without a password.')

pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ''
    text = re.sub(r'(?<=\w)-\s*\n\s*(?=\w)', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    pages.append({'page_number': page_number, 'text': text})

if not any(page['text'] for page in pages):
    raise ValueError('No extractable text was found in the source PDF.')

CHUNK_WORDS = 140
CHUNK_OVERLAP = 30
STEP_WORDS = CHUNK_WORDS - CHUNK_OVERLAP
chunks = []
for page in pages:
    words = page['text'].split()
    for start in range(0, len(words), STEP_WORDS):
        window = words[start:start + CHUNK_WORDS]
        if not window:
            continue
        chunks.append({
            'chunk_id': f"p{page['page_number']:03d}_c{start // STEP_WORDS + 1:02d}",
            'page_start': page['page_number'],
            'page_end': page['page_number'],
            'word_start': start,
            'word_end': start + len(window),
            'text': ' '.join(window),
        })

page_word_count = sum(len(page['text'].split()) for page in pages)
print(f'PDF pages: {len(pages)} | Extracted words: {page_word_count:,} | Chunks: {len(chunks):,}')
print('Chunking: page-preserving word windows; '
      f'{CHUNK_WORDS} words with {CHUNK_OVERLAP} words overlap.')
print('First chunk (actual extracted text):', chunks[0]['text'][:450])

PDF pages: 129 | Extracted words: 42,918 | Chunks: 457
Chunking: page-preserving word windows; 140 words with 30 words overlap.
First chunk (actual extracted text): NIST Special Publication NIST SP 800-63B-4 Digital Identity Guidelines Authentication and Authenticator Management David Temoshok James L. Fenton Yee-Yin Choong Naomi Lefkovitz Andrew Regenscheid Ryan Galluzzo Justin P . Richer This publication is available free of charge from: https://doi.org/10.6028/NIST.SP.800-63B-4


In [ ]:
from transformers.utils import logging as hf_logging

# Avoid progress-bar widget outputs that can remain stuck at 0% in saved notebooks.
hf_logging.disable_progress_bar()

embedder = SentenceTransformer(EMBEDDER_NAME)
reranker = CrossEncoder(RERANKER_NAME)
chunk_texts = [chunk['text'] for chunk in chunks]
chunk_embeddings = embedder.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)
print(f'Embedding model: {EMBEDDER_NAME} | Vector matrix: {chunk_embeddings.shape}')
print(f'Cross-encoder: {RERANKER_NAME}')

def baseline_search(query, top_k=10):
    started = time.perf_counter()
    query_embedding = embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    indices = np.argsort(similarities)[::-1][:top_k]
    results = []
    for rank, index in enumerate(indices, start=1):
        item = chunks[int(index)]
        results.append({
            **item,
            'baseline_rank': rank,
            'baseline_score': float(similarities[index]),
        })
    elapsed_ms = (time.perf_counter() - started) * 1000
    return results, elapsed_ms

def rerank_candidates(query, candidates):
    pairs = [(query, item['text']) for item in candidates]
    started = time.perf_counter()
    scores = reranker.predict(pairs, show_progress_bar=False)
    elapsed_ms = (time.perf_counter() - started) * 1000
    results = []
    for candidate, score in zip(candidates, scores):
        results.append({
            **candidate,
            'rerank_score': float(score),
        })
    results.sort(key=lambda item: item['rerank_score'], reverse=True)
    for rank, item in enumerate(results, start=1):
        item['rerank_rank'] = rank
    return results, elapsed_ms

In [4]:
# Each target anchor is a literal clause found in the source PDF. Its rank is
# measured after indexing; it is never used to alter retrieval or reranking.
query_specs = [
    {
        'name': 'AAL3 syncable-authenticator restriction',
        'query': 'Why are credentials synchronized between devices excluded from the highest authentication assurance level?',
        'target_pdf_page': 20,
        'evidence_anchor': 'syncable authenticators SHALL NOT be used at AAL3',
        'failure_type': 'constraint / negation',
    },
    {
        'name': 'Authenticator failure-attempt limit',
        'query': 'How many unsuccessful uses of one authenticator can occur on an account before the verifier disables that authenticator?',
        'target_pdf_page': 41,
        'evidence_anchor': 'no more than 100',
        'failure_type': 'buried numeric exception / constraint',
    },
    {
        'name': 'Short-secret attempt-counter behavior',
        'query': 'When a short authentication secret is replaced, does the account failed-attempt counter start over?',
        'target_pdf_page': 33,
        'evidence_anchor': 'Generating a new authentication secret SHALL NOT reset the failed authentication count',
        'failure_type': 'negation / exception',
    },
    {
        'name': 'Email-code exception',
        'query': 'Does the prohibition on email-based authentication also prohibit email address confirmation codes and recovery codes?',
        'target_pdf_page': 31,
        'evidence_anchor': 'not affected by the above prohibition',
        'failure_type': 'exception / keyword trap',
    },
    {
        'name': 'AAL2 phishing-resistant option',
        'query': 'At AAL2, what must the verifier make available regarding phishing resistance, and what is required of federal-agency staff?',
        'target_pdf_page': 18,
        'evidence_anchor': 'Verifiers SHALL offer at least one phishing-resistant authentication option at AAL2',
        'failure_type': 'role-specific constraint',
    },

]

query_results = []
for spec in query_specs:
    target_chunks = [
        chunk for chunk in chunks
        if chunk['page_start'] == spec['target_pdf_page']
        and re.search(re.escape(spec['evidence_anchor']), chunk['text'], re.IGNORECASE)
    ]
    if not target_chunks:
        raise ValueError(
            f"Grounding anchor not found on PDF page {spec['target_pdf_page']}: "
            f"{spec['evidence_anchor']}"
        )
    baseline, baseline_ms = baseline_search(spec['query'], top_k=10)
    reranked, rerank_ms = rerank_candidates(spec['query'], baseline)
    baseline_target_ranks = [x['baseline_rank'] for x in baseline if x['chunk_id'] in {t['chunk_id'] for t in target_chunks}]
    rerank_target_ranks = [x['rerank_rank'] for x in reranked if x['chunk_id'] in {t['chunk_id'] for t in target_chunks}]
    query_results.append({
        **spec,
        'target_chunk_ids': [item['chunk_id'] for item in target_chunks],
        'baseline': baseline,
        'reranked': reranked,
        'baseline_ms': baseline_ms,
        'rerank_ms': rerank_ms,
        'target_baseline_rank': min(baseline_target_ranks, default=None),
        'target_rerank_rank': min(rerank_target_ranks, default=None),
    })

print(f'Grounded queries evaluated: {len(query_results)}')
print('All evidence anchors were found verbatim in the recorded PDF page before ranking.')

Grounded queries evaluated: 5
All evidence anchors were found verbatim in the recorded PDF page before ranking.


In [5]:
rank_rows = []
for result in query_results:
    rank_rows.append({
        'query': result['name'],
        'target chunk(s)': ', '.join(result['target_chunk_ids']),
        'target page': result['target_pdf_page'],
        'target baseline rank': result['target_baseline_rank'],
        'target rerank rank': result['target_rerank_rank'],
        'baseline cosine': next((x['baseline_score'] for x in result['baseline'] if x['chunk_id'] in result['target_chunk_ids']), None),
        'cross-encoder score': next((x['rerank_score'] for x in result['reranked'] if x['chunk_id'] in result['target_chunk_ids']), None),
        'baseline top-3 has target': result['target_baseline_rank'] is not None and result['target_baseline_rank'] <= 3,
        'reranked top-3 has target': result['target_rerank_rank'] is not None and result['target_rerank_rank'] <= 3,
        'retrieval ms': result['baseline_ms'],
        'rerank ms': result['rerank_ms'],
    })
rank_table = pd.DataFrame(rank_rows)
print('Evidence-target rank changes, actual measured scores and stage timings:')
print(rank_table.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

failure_cases = [
    result for result in query_results
    if result['target_baseline_rank'] is not None
    and result['target_baseline_rank'] > 3
    and result['target_rerank_rank'] is not None
    and result['target_rerank_rank'] <= 3
]
print(f'\nGenuine baseline-miss / reranker-recovery cases: {len(failure_cases)}')
if failure_cases:
    for result in failure_cases:
        print(f"- {result['name']}: {result['target_chunk_ids']} "
              f"rank {result['target_baseline_rank']} -> {result['target_rerank_rank']} "
              f"(PDF page {result['target_pdf_page']})")
else:
    print('None observed in this run; do not claim a reranking recovery.')

for result in query_results:
    print(f"\nTOP-10 EVIDENCE: {result['name']} | {result['query']}")
    rows = []
    rerank_by_id = {item['chunk_id']: item for item in result['reranked']}
    for item in result['baseline']:
        reranked_item = rerank_by_id[item['chunk_id']]
        rows.append({
            'chunk': item['chunk_id'],
            'page': item['page_start'],
            'baseline rank': item['baseline_rank'],
            'cosine': item['baseline_score'],
            'rerank rank': reranked_item['rerank_rank'],
            'cross score': reranked_item['rerank_score'],
            'target evidence': item['chunk_id'] in result['target_chunk_ids'],
            'text excerpt': item['text'][:155],
        })
    print(pd.DataFrame(rows).to_string(index=False, float_format=lambda value: f'{value:.4f}'))
    print('Baseline selected top 3:', [(x['chunk_id'], x['page_start'], round(x['baseline_score'], 4)) for x in result['baseline'][:3]])
    print('Reranked selected top 3:', [(x['chunk_id'], x['page_start'], round(x['rerank_score'], 4)) for x in result['reranked'][:3]])

Evidence-target rank changes, actual measured scores and stage timings:
                                  query    target chunk(s)  target page  target baseline rank  target rerank rank  baseline cosine  cross-encoder score  baseline top-3 has target  reranked top-3 has target  retrieval ms  rerank ms
AAL3 syncable-authenticator restriction           p020_c01           20                   NaN                 NaN              NaN                  NaN                      False                      False       67.1584   230.4013
    Authenticator failure-attempt limit           p041_c02           41                2.0000              1.0000           0.7295               4.1853                       True                       True       15.3808   218.6772
  Short-secret attempt-counter behavior p033_c02, p033_c03           33                4.0000              1.0000           0.5142               0.5854                      False                       True       12.0278   216.3373
    

In [ ]:
# Generate paired answers only after retrieval/reranking. The prompt template and
# decoding settings are identical for both pipelines; no answer is hardcoded.
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_NAME)
generator = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_NAME)
PROMPT_TEMPLATE = (
    'Answer the question using only the context below. Give a direct, concise answer '
    'and preserve the relevant condition. If the answer is absent, say the context '
    'does not specify.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:'
)
GENERATION_SETTINGS = {
    'max_new_tokens': 64,
    'do_sample': False,
    'num_beams': 4,
    'no_repeat_ngram_size': 3,
    'repetition_penalty': 1.1,
}

CONTEXT_TOKEN_BUDGETS = (200, 100, 100)

def generate_answer(question, selected_chunks):
    if len(selected_chunks) != 3:
        raise ValueError(f'Expected exactly three selected chunks, received {len(selected_chunks)}.')
    started = time.perf_counter()
    context_blocks = []
    for rank, item in enumerate(selected_chunks):
        token_ids = tokenizer(item['text'], add_special_tokens=False)['input_ids']
        excerpt = tokenizer.decode(
            token_ids[:CONTEXT_TOKEN_BUDGETS[rank]], skip_special_tokens=True
        )
        context_blocks.append(
            f"[PDF page {item['page_start']}; chunk {item['chunk_id']}] {excerpt}"
        )
    context = '\n'.join(context_blocks)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=False)
    input_tokens = inputs['input_ids'].shape[1]
    if input_tokens > 512:
        raise ValueError(
            f'Selected top-3 excerpts plus prompt is {input_tokens} tokens; '
            'the 512-token FLAN-T5 input limit was exceeded, so nothing was truncated.'
        )
    output = generator.generate(**inputs, **GENERATION_SETTINGS)
    answer = tokenizer.decode(output[0], skip_special_tokens=True).strip()
    elapsed_ms = (time.perf_counter() - started) * 1000
    if not answer:
        raise RuntimeError('Local generator returned an empty answer; no substitute was used.')
    return answer, elapsed_ms, input_tokens


# Compare both pipelines on up to two verified failure queries. If fewer than two
# qualify, use the first grounded queries and report the shortfall without relabeling it.
answer_cases = failure_cases[:2]
if len(answer_cases) < 2:
    remaining = [result for result in query_results if result not in answer_cases]
    answer_cases.extend(remaining[:2 - len(answer_cases)])
answer_rows = []
for result in answer_cases:
    p1_started = time.perf_counter()
    p1_candidates, p1_retrieval_ms = baseline_search(result['query'], top_k=10)
    if [x['chunk_id'] for x in p1_candidates] != [x['chunk_id'] for x in result['baseline']]:
        raise RuntimeError('Repeated baseline query produced different candidates; paired comparison is invalid.')
    p1_answer, p1_generation_ms, p1_input_tokens = generate_answer(result['query'], p1_candidates[:3])
    p1_e2e_ms = (time.perf_counter() - p1_started) * 1000

    p2_started = time.perf_counter()
    p2_candidates, p2_retrieval_ms = baseline_search(result['query'], top_k=10)
    if [x['chunk_id'] for x in p2_candidates] != [x['chunk_id'] for x in result['baseline']]:
        raise RuntimeError('Pipeline 2 did not receive the same top-10 baseline candidates.')
    p2_ranked, p2_rerank_ms = rerank_candidates(result['query'], p2_candidates)
    p2_answer, p2_generation_ms, p2_input_tokens = generate_answer(result['query'], p2_ranked[:3])
    p2_e2e_ms = (time.perf_counter() - p2_started) * 1000

    answer_rows.append({
        'query': result['name'],
        'pipeline1_chunks': [x['chunk_id'] for x in p1_candidates[:3]],
        'pipeline1_answer': p1_answer,
        'pipeline1_retrieval_ms': p1_retrieval_ms,
        'pipeline1_generation_ms': p1_generation_ms,
        'pipeline1_input_tokens': p1_input_tokens,
        'pipeline1_e2e_ms': p1_e2e_ms,
        'pipeline2_chunks': [x['chunk_id'] for x in p2_ranked[:3]],
        'pipeline2_answer': p2_answer,
        'pipeline2_retrieval_ms': p2_retrieval_ms,
        'pipeline2_rerank_ms': p2_rerank_ms,
        'pipeline2_generation_ms': p2_generation_ms,
        'pipeline2_input_tokens': p2_input_tokens,
        'pipeline2_e2e_ms': p2_e2e_ms,
    })
    print(f"\nANSWER COMPARISON: {result['name']}")
    print('Pipeline 1 selected:', answer_rows[-1]['pipeline1_chunks'])
    print(f"Pipeline 1 answer (local LLM, {p1_input_tokens} input tokens):", p1_answer)
    print('Pipeline 2 selected:', answer_rows[-1]['pipeline2_chunks'])
    print(f"Pipeline 2 answer (same local LLM and prompt, {p2_input_tokens} input tokens):", p2_answer)

# Keep retrieval/reranking timings separate from generation timings.
# Generation was measured only for the two answer-comparison queries, so combining
# both sets into one table would create misleading NaN cells for the other queries.
retrieval_latency_rows = []
for result in query_results:
    retrieval_latency_rows.append({
        'query': result['name'],
        'baseline retrieval (ms)': result['baseline_ms'],
        'cross-encoder rerank (ms)': result['rerank_ms'],
    })

retrieval_latency_table = pd.DataFrame(retrieval_latency_rows)
print('\\nRetrieval and re-ranking latency (milliseconds; this run only):')
print(retrieval_latency_table.to_string(
    index=False, float_format=lambda value: f'{value:.2f}'
))

generation_latency_rows = []
for row in answer_rows:
    generation_latency_rows.append({
        'query': row['query'],
        'baseline generation (ms)': row['pipeline1_generation_ms'],
        'reranked generation (ms)': row['pipeline2_generation_ms'],
        'baseline end-to-end (ms)': row['pipeline1_e2e_ms'],
        'reranked end-to-end (ms)': row['pipeline2_e2e_ms'],
    })

generation_latency_table = pd.DataFrame(generation_latency_rows)
print('\\nGeneration and end-to-end latency (milliseconds; answer-comparison queries only):')
print(generation_latency_table.to_string(
    index=False, float_format=lambda value: f'{value:.2f}'
))

print(f"\\nGenerator model: {GENERATOR_NAME}")
print(f"Identical prompt template: {PROMPT_TEMPLATE!r}")
print(f"Generation settings: {GENERATION_SETTINGS}")
